In [1]:
import pandas as pd
from pathlib import Path

In [20]:
path_barras_db = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\reporte_barras.xlsx")
path_estaciones_db = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\reporte_subestaciones.xlsx")
dfb = pd.read_excel(path_barras_db, skiprows = 6)
dfe = pd.read_excel(path_estaciones_db, skiprows = 6)
to_save_path = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones")

In [3]:
dfb.columns

Index(['ID', 'Nombre', 'Nombre Centro Control', 'Nombre Propietario',
       'Nombre Coordinado', 'Nombre Subestación', 'Número', 'Nemotecnico',
       'Descripcion', 'Tipo de barra (cable, tubo, GIS, switchgear, etc.)',
       '18.1 Limite térmico permanente',
       '18.2 Capacidad nominal corriente cortocircuito, de duración  de 1 [s] o bien a 3 [s] de las barras',
       '18.3 Conductor tipo 1 (AAAC, AASC, ACAR, ETC.)',
       '18.4 Sección del conductor', '18.5 Número de conductores por fase',
       '18.6 Fecha de entrada en operación',
       '1 Informe y tabla de capacidad térmica del conductor, en función de las T° amb. y del conductor (tabla relación corriente-T°)'],
      dtype='str')

In [4]:
dfb2 = dfb[dfb.columns[:9]]
dfb2.drop(columns= "Descripcion", errors = "ignore")
dfb3 = dfb2.rename(columns = {'Tipo de barra (cable, tubo, GIS, switchgear, etc.)': "Tipo de barra"})


In [5]:
dfe.columns[:12]

Index(['ID', 'Nombre', 'Nombre Centro Control', 'Nombre Propietario',
       'Nombre Coordinado', 'Número', 'Nemotecnico', 'Descripcion', 'Región',
       'Provincia', 'Comuna',
       '5.1 Identificar patios por nivel de tensión.  (Artículo 19 Anexo Técnico 03/2025)'],
      dtype='str')

In [6]:
dfe2 = dfe[["ID", "Nombre", "Número", "Nemotecnico", 'Región',
       'Provincia', 'Comuna']]
dfe3 = dfe2.rename(columns={"ID": "ID_E", "Nombre": "Nombre_E", "Número": "Número_E", "Nemotecnico": "Nemotecnico_E"})

In [8]:
dfe4 = dfe3.set_index("Nombre_E")
dfb4 = dfb3.set_index("Nombre Subestación")

In [9]:
dfv = pd.merge(left = dfb4, right = dfe4, left_index = True, right_index = True, how = "left").reset_index()

In [12]:
df = dfv[~dfv["Región"].isna()]

In [13]:
df.head()

,Nombre Subestación,ID,Nombre,Nombre Centro Control,Nombre Propietario,Nombre Coordinado,Número,Nemotecnico,Descripcion,ID_E,Número_E,Nemotecnico_E,Región,Provincia,Comuna
0,S/E CENTRAL ALFALFAL,1,BA S/E CENTRAL ALFALFAL 12KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo
1,S/E CENTRAL ALFALFAL,2,BA S/E CENTRAL ALFALFAL 12KV BP2,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo
2,S/E CENTRAL MAITENES,6,BA S/E CENTRAL MAITENES 6.6KV B1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE004G0010,NaN,201,4,SE004G0010,Metropolitana de Santiago,Cordillera,San José de Maipo
3,S/E CENTRAL QUELTEHUES,7,BA S/E CENTRAL QUELTEHUES 110KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo
4,S/E CENTRAL QUELTEHUES,8,BA S/E CENTRAL QUELTEHUES 12KV,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo


In [14]:
df["Región"].unique()

<ArrowStringArray>
[                'Metropolitana de Santiago',
                                  'Los Ríos',
                                    'Biobío',
                                     'Ñuble',
                                     'Maule',
       'Libertador Gral. Bernardo O'Higgins',
                                 'Los Lagos',
                                   'Atacama',
                                'Valparaíso',
                                  'Coquimbo',
                              'La Araucanía',
                               'Antofagasta',
                                  'Tarapacá',
                        'Arica y Parinacota',
      'Magallanes y de la Antártica Chilena',
 'Aysén del General Carlos Ibáñez del Campo']
Length: 16, dtype: str

In [17]:
# Rescatado del banco central
macrozonas = {
    # 1.- Macrozona Norte
    'Arica y Parinacota': 'Norte',
    'Tarapacá': 'Norte',
    'Antofagasta': 'Norte',
    'Atacama': 'Norte',
    
    # 2.- Macrozona Centro 
    'Coquimbo': 'Centro',
    'Valparaíso': 'Centro',
    'Metropolitana de Santiago': 'Centro', # En realidad esta es una aparte pero creo que es suficiente si es agrupada
    
    # 4.- Macrozona Centro Sur
    'Libertador Gral. Bernardo O\'Higgins': 'Centro Sur',
    'Maule': 'Centro Sur',
    'Ñuble': 'Centro Sur',
    'Biobío': 'Centro Sur',
    
    # 5.- Macrozona Sur
    'La Araucanía': 'Sur',
    'Los Ríos': 'Sur',
    'Los Lagos': 'Sur',
    
    # 6.- Macrozona Austral
    'Aysén del General Carlos Ibáñez del Campo': 'Austral',
    'Magallanes y de la Antártica Chilena': 'Austral'
}

In [18]:
df['Macrozona'] = df['Región'].map(macrozonas)

In [21]:
df.to_csv(to_save_path/"LocatedBars.csv", sep=";", encoding="utf-8", index=False)